# Práctica guiada — Inferencia y contratos

Este notebook es para el alumnado. Completa las celdas marcadas con `TODO` y responde las preguntas antes de mirar la solución docente.

**Objetivo:** pasar de una fila CSV a una especificación clara para el módulo de inferencia de la clase 2.

## Cómo trabajar

1. Copia esta carpeta como tu espacio de trabajo.
2. Ejecuta `uv sync` desde `problem/starter/` cuando empiece la clase 2.
3. En esta práctica no implementes todavía el CLI completo.
4. Trabaja en parejas y cambia quién escribe cada diez minutos.
5. No abras `solutions/` hasta el debrief.

In [1]:
from dataclasses import dataclass
import csv
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError

DATA_PATH = Path("../../../../../../assets/03-wine-quality/inference_samples.csv")
print(DATA_PATH)
print(DATA_PATH.exists())

..\..\..\..\..\..\assets\03-wine-quality\inference_samples.csv
True


## 1. Leer los datos nuevos

### Práctica A — Inspección del CSV

Completa la celda siguiente para:

- Leer el CSV con `csv.DictReader`.
- Imprimir el número de filas.
- Imprimir las columnas.
- Guardar la primera fila en una variable llamada `row`.

In [2]:
# TODO: implementa la lectura del CSV.
# Pista: usa DATA_PATH.open(newline="", encoding="utf-8")
data = DATA_PATH.open(newline="", encoding="utf-8", )

rows = []
row = {}

for i in data:
    print(i)
    rows.append(i)

sample_id,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol

red-001,7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4

red-002,7.8,0.88,0.0,2.6,0.098,25.0,67.0,0.9968,3.2,0.68,9.8

red-003,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.998,3.16,0.58,9.8

red-004,7.3,0.65,0.0,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0

red-005,7.8,0.58,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5



In [ ]:
# Columnas
columnas = rows[0]
print(columnas)

sample_id,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol



In [4]:
rows.pop(0)
rows

['red-001,7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4\r\n',
 'red-002,7.8,0.88,0.0,2.6,0.098,25.0,67.0,0.9968,3.2,0.68,9.8\r\n',
 'red-003,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.998,3.16,0.58,9.8\r\n',
 'red-004,7.3,0.65,0.0,1.2,0.065,15.0,21.0,0.9946,3.39,0.47,10.0\r\n',
 'red-005,7.8,0.58,0.02,2.0,0.073,9.0,18.0,0.9968,3.36,0.57,9.5\r\n']

In [5]:
for i in range(len(rows)):
    rows[i] = rows[i].rstrip("\r\n")

In [6]:
# Numero de filas en rows
len(rows)

5

In [ ]:
# Row con la informacion de la primera fila
row = rows[0]
print(row)

red-001,7.4,0.7,0.0,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4


### Preguntas

1. ¿Qué campo identifica la muestra pero no es una característica del modelo?

sample_id no hace falta para el modelo y es el identificador de cada muestra

2. ¿Cuántas características numéricas hay?

El resto de las columnas: fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol

3. ¿Qué columna extra añadirías para probar el contrato?

Cualquier columna extra debería devolver un error si se añade como condición que no puedan entrar columnas extras (forbid). Incluso incluir el id a la hora de pasar el contrato debería dar un error.

4. ¿Qué valor de `ph` sería claramente inválido?

Cualquier valor menor a 0 o superior a 14, aunque tratando vinos se puede llegar a acotar hasta rangos como de 2 a 5.

## 2. Construir el contrato con Pydantic

El contrato del modelo contiene estas características:

| Campo | Rango orientativo |
|---|---|
| `fixed_acidity` | 0–20 |
| `volatile_acidity` | 0–2 |
| `citric_acid` | 0–2 |
| `residual_sugar` | 0–20 |
| `chlorides` | 0–1 |
| `free_sulfur_dioxide` | 0–100 |
| `total_sulfur_dioxide` | 0–300 |
| `density` | 0.98–1.01 |
| `ph` | 2.5–4.5 |
| `sulphates` | 0–3 |
| `alcohol` | 5–20 |

In [8]:
class WineQualityRequestPydantic(BaseModel):
    # TODO: prohíbe campos extra con model_config.
    model_config = ConfigDict(extra="forbid")
    # TODO: declara las 11 características con Field(ge=..., le=...).
    fixed_acidity: float = Field(ge=0, le=20)
    volatile_acidity: float = Field(ge=0, le=2)
    citric_acid: float = Field(ge=0, le=2)
    residual_sugar: float = Field(ge=0, le=20)
    chlorides: float = Field(ge=0, le=1)
    free_sulfur_dioxide: float = Field(ge=0, le=100)
    total_sulfur_dioxide: float = Field(ge=0, le=300)
    density: float = Field(ge=0.98, le=1.01)
    ph: float = Field(ge=2.5, le=4.5)
    sulphates: float = Field(ge=0, le=3)
    alcohol: float = Field(ge=5, le=20)
    pass

In [9]:
# TODO: separa sample_id de la fila y valida solo las características del modelo.
# model_row = ...
# request = WineQualityRequestPydantic.model_validate(model_row)
columnas = [
    'fixed_acidity',
    'volatile_acidity',
    'citric_acid',
    'residual_sugar',
    'chlorides',
    'free_sulfur_dioxide',
    'total_sulfur_dioxide',
    'density',
    'ph',
    'sulphates',
    'alcohol'
]
valor_id, valores_columnas = rows[0].split(',', 1)
valores_row0 = [float(x) for x in valores_columnas.split(',')]
model_row = dict(zip(columnas, valores_row0))
print(model_row)

{'fixed_acidity': 7.4, 'volatile_acidity': 0.7, 'citric_acid': 0.0, 'residual_sugar': 1.9, 'chlorides': 0.076, 'free_sulfur_dioxide': 11.0, 'total_sulfur_dioxide': 34.0, 'density': 0.9978, 'ph': 3.51, 'sulphates': 0.56, 'alcohol': 9.4}


In [10]:
request = WineQualityRequestPydantic.model_validate(model_row)

### Práctica B — Validación

Completa una prueba para cada caso:

- Una fila válida.
- Una columna `unexpected_field`.
- Un `ph` fuera de rango.

Antes de ejecutar, escribe qué error esperas.

In [11]:
# TODO: completa los casos y captura ValidationError.
# try:
#     ...
# except ValidationError as error:
#     print(error)

datos_validos = {
    "fixed_acidity": 7.4,
    "volatile_acidity": 0.7,
    "citric_acid": 0.0,
    "residual_sugar": 1.9,
    "chlorides": 0.076,
    "free_sulfur_dioxide": 11.0,
    "total_sulfur_dioxide": 34.0,
    "density": 0.9978,
    "ph": 3.51,
    "sulphates": 0.56,
    "alcohol": 9.4
}
try:
    modelo_valido = WineQualityRequestPydantic(**datos_validos)
    print("Validación completa")
except ValidationError as error:
    print(error)

Validación completa


In [12]:
datos_con_extra = datos_validos.copy()
datos_con_extra["unexpected_field"] = 123
try:
    WineQualityRequestPydantic(**datos_con_extra)
except ValidationError as error:
    print("Error:\n", error)

Error:
 1 validation error for WineQualityRequestPydantic
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


In [13]:
datos_ph_invalido = datos_validos.copy()
datos_ph_invalido["ph"] = 5.5
try:
    WineQualityRequestPydantic(**datos_ph_invalido)
except ValidationError as error:
    print("Error:\n", error)

Error:
 1 validation error for WineQualityRequestPydantic
ph
  Input should be less than or equal to 4.5 [type=less_than_equal, input_value=5.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## 3. Comparar con `dataclass`

Construye la misma estructura con `dataclass`. No añadas todavía validación manual: observa qué garantiza Python y qué no garantiza.

In [14]:
@dataclass(frozen=True, slots=True)
class WineQualityRequestDataclass:
    # TODO: declara las mismas 11 características.
    fixed_acidity: float
    volatile_acidity: float
    citric_acid: float
    residual_sugar: float
    chlorides: float
    free_sulfur_dioxide: float
    total_sulfur_dioxide: float
    density: float
    ph: float
    sulphates: float
    alcohol: float
    pass

# TODO: crea una instancia válida y prueba qué ocurre con ph=99.0.
instancia_valida = WineQualityRequestDataclass(
    fixed_acidity=7.4,
    volatile_acidity=0.7,
    citric_acid=0.0,
    residual_sugar=1.9,
    chlorides=0.076,
    free_sulfur_dioxide=11.0,
    total_sulfur_dioxide=34.0,
    density=0.9978,
    ph=3.51,
    sulphates=0.56,
    alcohol=9.4
)
instancia_ph_invalido = WineQualityRequestDataclass(
    fixed_acidity=7.4,
    volatile_acidity=0.7,
    citric_acid=0.0,
    residual_sugar=1.9,
    chlorides=0.076,
    free_sulfur_dioxide=11.0,
    total_sulfur_dioxide=34.0,
    density=0.9978,
    ph=99.0,
    sulphates=0.56,
    alcohol=9.4
)

In [15]:
print(instancia_valida)
print(instancia_ph_invalido)

WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=3.51, sulphates=0.56, alcohol=9.4)
WineQualityRequestDataclass(fixed_acidity=7.4, volatile_acidity=0.7, citric_acid=0.0, residual_sugar=1.9, chlorides=0.076, free_sulfur_dioxide=11.0, total_sulfur_dioxide=34.0, density=0.9978, ph=99.0, sulphates=0.56, alcohol=9.4)


### Decisión de diseño

Completa la frase:

> Usaría Pydantic en los datos de entrada que estén recién importados para asegurar que cumplan condiciones específicas. Usaría `dataclass` en datos que ya hayan pasado por pasos anteriores en el código porque no haría falta tener tantas restricciones que revisar y puede ser más ligero.

## 4. Definir el orden del vector

El modelo no recibe un diccionario. Recibe una lista numérica con un orden estable.

In [16]:
# TODO: rellena el orden exacto a partir del CSV y del contrato.
FEATURE_NAMES = (
    # "fixed_acidity",
    # ...
    'fixed_acidity',
    'volatile_acidity',
    'citric_acid',
    'residual_sugar',
    'chlorides',
    'free_sulfur_dioxide',
    'total_sulfur_dioxide',
    'density',
    'ph',
    'sulphates',
    'alcohol'
)

# TODO: construye vector usando getattr(request, name).
# vector = [...]
vector = [getattr(request, name) for name in FEATURE_NAMES]
print(vector)


[7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]


### Práctica C — Error de orden

Intercambia mentalmente `density` y `alcohol`. Responde:

- ¿El vector sigue teniendo 11 valores?

Si se mantendrían 11 valores.

- ¿El clasificador detectaría necesariamente el error?

Si saltaría un error tanto por no tener el orden correcto de las variables al entrar al modelo como por los rangos que no se pueden cumplir ambos al mismo tiempo.

- ¿Cómo evitarías que cada script definiera un orden distinto?

Utilizando la variable global de FEATURE_NAMES en cada uno de los scripts donde se requiera para las funciones, también se podría dar dentro de cada función haciendo que solamente se tenga que declarar en el script que incluya el main y pasarlo a las funciones necesarias.

## 5. Contrato de salida

Diseña la estructura que devolverá el módulo. Debe permitir responder:

- ¿Qué muestra se ha procesado?

El valor de sample_id.

- ¿Qué categoría se ha predicho?

La calidad del vino (quality_band)

- ¿Con qué confianza?

El modelo devuelve una probabilidad entre 0 y 1 (confidence)

- ¿Qué versión de modelo y preprocesado se utilizaron?

Los valores de model_version y preprocessing_version

In [17]:
class WineQualityPredictionPydantic(BaseModel):
    # TODO: declara los campos de salida y sus restricciones.
    sample_id: str
    quality_band: Literal["needs_review", "acceptable", "excellent"]
    confidence: float = Field(ge=0, le=1)
    model_version: str
    preprocessing_version: str
    pass

# TODO: escribe un ejemplo de salida para red-001.
salida_red_001 = WineQualityPredictionPydantic(
    sample_id="red-001",
    quality_band="excellent",
    confidence=0.85,
    model_version="1.0.0",
    preprocessing_version="1.2.0"
)

## 6. Demo del docente: observar la inferencia

El docente ejecutará el notebook resuelto y el comando de referencia. Mientras tanto, anota el recorrido:

```text
CSV → contrato → preprocesado → modelo cargado → predicción
```

No copies la implementación. Tu código de producción empieza en la clase 2 con el starter.

## Entrega

Entrega una captura o exportación con:

- La tabla de contrato.
- Las respuestas de las prácticas A, B y C.
- El orden de las 11 características.
- Un ejemplo de entrada inválida.
- Un contrato de salida.
- Tu plan de implementación para `contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`.

Para la implementación, estarían todos los archivos mencionados (`contracts.py`, `preprocess.py`, `inference.py` y `predict_file.py`) dentro del src.

`contracts.py`: Tendría las clases de comprobación de la entrada y la salida
- `WineQualityPredictionPydantic(BaseModel)`
- `WineQualityRequestPydantic(BaseModel)`

`preprocess.py`: Aquí se encuentran las funciones para limpiar datos antes de asignarlos a  la clase `WineQualityRequestPydantic(BaseModel)` desde los datos de entrada importador con una función de `cargar(path)` para transformarlos en el vector que puede entrar al modelo. También puede incluir el `FEATURE_NAMES` como variable que guarda el orden de las features.

`inference.py`: Dentro de este Script se pueden usar funciones para:
- cargar el modelo
- usar el modelo para inferir los valores de salida

`predict_file.py`: Este Script puede tener el main donde se van a llamar a las funciones necesarias en orden para llevar acabo las predicciones y generar un archivo de salida con las predicciones.